<a href="https://colab.research.google.com/github/exdsgift/NerGuard/blob/main/PII_detector.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Importing the modules

In [1]:
!pip install -q "transformers>=4.40" datasets seqeval accelerate

import torch
import os, json, unicodedata
from datasets import load_dataset, DatasetDict
import numpy as np
from typing import List, Dict, Tuple
from datasets import load_dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForTokenClassification
from transformers import DataCollatorWithPadding, TrainingArguments, Trainer
from seqeval.metrics import f1_score, classification_report
from seqeval.scheme import IOB2
import pprint as pp

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


#Specifying CUDA as the device for torch

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
!nvidia-smi

Sun Oct 19 06:40:01 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8             10W /   70W |       2MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# Loading dataset

In [3]:
ds = load_dataset("gretelai/synthetic_pii_finance_multilingual")
print(ds)

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/48.4M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/5.42M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/50346 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5594 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['level_0', 'index', 'document_type', 'document_description', 'expanded_type', 'expanded_description', 'language', 'language_description', 'domain', 'generated_text', 'pii_spans', 'conformance_score', 'quality_score', 'toxicity_score', 'bias_score', 'groundedness_score'],
        num_rows: 50346
    })
    test: Dataset({
        features: ['level_0', 'index', 'document_type', 'document_description', 'expanded_type', 'expanded_description', 'language', 'language_description', 'domain', 'generated_text', 'pii_spans', 'conformance_score', 'quality_score', 'toxicity_score', 'bias_score', 'groundedness_score'],
        num_rows: 5594
    })
})


In [4]:
if "validation" not in ds:
    tmp = ds["train"].train_test_split(test_size=0.1, seed=42)
    ds = DatasetDict({
        "train": ds["train"],
        "validation": tmp["test"],
        "test": tmp["train"]
    })
print(ds)

DatasetDict({
    train: Dataset({
        features: ['level_0', 'index', 'document_type', 'document_description', 'expanded_type', 'expanded_description', 'language', 'language_description', 'domain', 'generated_text', 'pii_spans', 'conformance_score', 'quality_score', 'toxicity_score', 'bias_score', 'groundedness_score'],
        num_rows: 50346
    })
    validation: Dataset({
        features: ['level_0', 'index', 'document_type', 'document_description', 'expanded_type', 'expanded_description', 'language', 'language_description', 'domain', 'generated_text', 'pii_spans', 'conformance_score', 'quality_score', 'toxicity_score', 'bias_score', 'groundedness_score'],
        num_rows: 5035
    })
    test: Dataset({
        features: ['level_0', 'index', 'document_type', 'document_description', 'expanded_type', 'expanded_description', 'language', 'language_description', 'domain', 'generated_text', 'pii_spans', 'conformance_score', 'quality_score', 'toxicity_score', 'bias_score', 'grounde

In [5]:
ds.shape

{'train': (50346, 16), 'validation': (5035, 16), 'test': (45311, 16)}

In [6]:
df = ds["train"].to_pandas()
df.head()

,level_0,index,document_type,document_description,expanded_type,expanded_description,language,language_description,domain,generated_text,pii_spans,conformance_score,quality_score,toxicity_score,bias_score,groundedness_score
0,40012,40012,Supply Chain Management Agreement,A legal contract outlining the terms and condi...,Vendor Management Contract,This subtype involves the contractual agreemen...,English,English language as spoken in the United State...,finance,SUPPLY CHAIN MANAGEMENT AGREEMENT\n\nThis Supp...,"[{""start"": 119, ""end"": 141, ""label"": ""date""}, ...",85,90,5,15,95
1,46425,46425,Supply Chain Management Agreement,A legal contract outlining the terms and condi...,Supply Chain Resilience Framework,This subtype details the framework for buildin...,English,English language as spoken in the United State...,finance,SUPPLY CHAIN RESILIENCE FRAMEWORK\n\nThis Supp...,"[{""start"": 119, ""end"": 142, ""label"": ""date""}, ...",92,87,5,12,95
2,4689,4689,Real Estate Loan Agreement,A legal contract outlining terms and condition...,International Real Estate Investment Loan Agre...,This subtype encompasses loans for internation...,Spanish,Spanish language as spoken in Spain or Mexico,finance,CONTRATO DE PRÉSTAMO PARA INVERSIÓN INMOBILIAR...,"[{""start"": 182, ""end"": 209, ""label"": ""street_a...",85,90,5,15,95
3,3002,3002,Real Estate Loan Agreement,A legal contract outlining terms and condition...,Commercial Property Loan Contract,This subtype focuses on loans for commercial r...,Italian,Italian language as spoken in Italy,finance,REPUBBLICA ITALIANA\n\nCONTRATTO DI PRESTITO I...,"[{""start"": 85, ""end"": 103, ""label"": ""name""}, {...",85,90,5,10,95
4,16187,16187,Email,A communication sent electronically containing...,Invitation,Create an email inviting recipients to an even...,France,French language as spoken in France,finance,Subject: Invitation à notre soirée de lancemen...,"[{""start"": 202, ""end"": 207, ""label"": ""time""}, ...",85,90,5,15,95


#Activating the mdeberta-v3-base tokenizer (fast) and support functions

In [7]:
try:
  tok = AutoTokenizer.from_pretrained("microsoft/mdeberta-v3-base", use_fast=True)
  assert tok.is_fast, "Serve tokenizer fast per offset_mapping"
except Exception as e:
  print("An error occurred while downloading the tokenizer.")
  print(str(e))
  import traceback
  print(traceback.format_exc())

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:564: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


In [8]:
def parse_spans(spans):
    if isinstance(spans, str):
        try:
            spans = json.loads(spans)
        except json.JSONDecodeError:
            return []
    if not isinstance(spans, list):
        return []
    out = []
    for s in spans:
        if isinstance(s, dict) and all(k in s for k in ("start","end","label")):
            out.append(s)
    return out

In [9]:
def collect_label_set(dataset_dict):
    labels = set()
    for split in dataset_dict.keys():
        for ex in dataset_dict[split]:
            for s in parse_spans(ex["pii_spans"]):
                labels.add(s["label"])
    return sorted(labels)

entity_labels = collect_label_set(ds)  # es: ['company','date','name','street_address', ...]
bio_labels = ["O"] + [f"B-{l}" for l in entity_labels] + [f"I-{l}" for l in entity_labels]
label2id = {l:i for i,l in enumerate(bio_labels)}
id2label = {i:l for l,i in label2id.items()}

In [10]:
bio_labels

['O',
 'B-account_pin',
 'B-api_key',
 'B-bank_routing_number',
 'B-bban',
 'B-company',
 'B-credit_card_number',
 'B-credit_card_security_code',
 'B-customer_id',
 'B-date',
 'B-date_of_birth',
 'B-date_time',
 'B-driver_license_number',
 'B-email',
 'B-employee_id',
 'B-first_name',
 'B-iban',
 'B-ipv4',
 'B-ipv6',
 'B-last_name',
 'B-local_latlng',
 'B-name',
 'B-passport_number',
 'B-password',
 'B-phone_number',
 'B-ssn',
 'B-street_address',
 'B-swift_bic_code',
 'B-time',
 'B-user_name',
 'I-account_pin',
 'I-api_key',
 'I-bank_routing_number',
 'I-bban',
 'I-company',
 'I-credit_card_number',
 'I-credit_card_security_code',
 'I-customer_id',
 'I-date',
 'I-date_of_birth',
 'I-date_time',
 'I-driver_license_number',
 'I-email',
 'I-employee_id',
 'I-first_name',
 'I-iban',
 'I-ipv4',
 'I-ipv6',
 'I-last_name',
 'I-local_latlng',
 'I-name',
 'I-passport_number',
 'I-password',
 'I-phone_number',
 'I-ssn',
 'I-street_address',
 'I-swift_bic_code',
 'I-time',
 'I-user_name']

spans → BIO with offset_mapping

In [11]:
def spans_to_bio(text, spans, label2id, tokenizer, max_length=256):
    import unicodedata
    text = unicodedata.normalize("NFC", text)
    spans = parse_spans(spans)

    # 1) maschera char-level
    char_tags = ["O"] * len(text)
    for s in spans:
        start, end, lab = s.get("start"), s.get("end"), s.get("label")
        if not isinstance(start,int) or not isinstance(end,int) or not isinstance(lab,str):
            continue
        if start < 0 or end <= start or end > len(text):
            continue
        if f"B-{lab}" not in label2id or f"I-{lab}" not in label2id:
            continue
        char_tags[start] = f"B-{lab}"
        for i in range(start+1, end):
            char_tags[i] = f"I-{lab}"

    # 2) tokenizza con special e offsets
    enc = tokenizer(
        text,
        return_offsets_mapping=True,
        return_attention_mask=True,
        add_special_tokens=True,     # <-- importante
        truncation=True,
        max_length=max_length
    )
    offsets = enc["offset_mapping"]

    tags = []
    for (a, b) in offsets:
        if a == 0 and b == 0:
            tags.append(None)        # special token -> ignora in loss
            continue
        if a == b:
            tags.append("O")
            continue
        start_tag = char_tags[a] if 0 <= a < len(text) else "O"
        if start_tag.startswith("B-"):
            lab = start_tag
        else:
            window = [t for t in char_tags[a:b] if t != "O"]
            if window:
                any_lab = window[0]
                lab = "I-" + any_lab.split("-", 1)[1]
            else:
                lab = "O"
        tags.append(lab)

    ignore_index = -100
    label_ids = [ignore_index if t is None else label2id[t] for t in tags]
    return enc["input_ids"], enc["attention_mask"], label_ids


Processing all splits

In [12]:
from transformers import DataCollatorForTokenClassification

TEXT_COL = "generated_text"
SPAN_COL = "pii_spans"

def preprocess_batch(batch):
    input_ids, attention_mask, labels = [], [], []
    for text, spans in zip(batch[TEXT_COL], batch[SPAN_COL]):
        ids, mask, y = spans_to_bio(text, spans, label2id, tok, max_length=256)
        input_ids.append(ids); attention_mask.append(mask); labels.append(y)
    return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}

remove_cols = list(ds["train"].column_names)
processed = DatasetDict()
for split in ds.keys():
    processed[split] = ds[split].map(
        preprocess_batch,
        batched=True,
        remove_columns=remove_cols
    ).with_format("torch",
                  columns=["input_ids","attention_mask"],
                  output_all_columns=True)

#.with_format("torch", columns=["input_ids","attention_mask","labels"])
train_processed = processed["train"]
val_processed   = processed["validation"]

data_collator = DataCollatorForTokenClassification(
    tokenizer=tok,
    padding=True,
    label_pad_token_id=-100
)

Map:   0%|          | 0/50346 [00:00<?, ? examples/s]

Map:   0%|          | 0/5035 [00:00<?, ? examples/s]

Map:   0%|          | 0/45311 [00:00<?, ? examples/s]

In [13]:
ds = train_processed.to_pandas()
ds.head(3)

,input_ids,attention_mask,labels
0,"[1, 260, 198259, 12970, 35332, 9390, 123776, 6...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[-100, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
1,"[1, 260, 198259, 12970, 35332, 9390, 70930, 41...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[-100, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
2,"[1, 121147, 11162, 1451, 364, 81081, 229860, 6...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[-100, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."


In [14]:
print(train_processed[0])

{'input_ids': tensor([     1,    260, 198259,  12970,  35332,   9390, 123776,    622,  65851,
           260, 231124,  65851,   1495,    260,  57526,  57197,   8477,    260,
         92537,    275,   1760,    314,  92537,   5806,    340,    260, 106033,
          2388,    528,    305,    715,    334,    706,   3118,    305,   3619,
           262,  26222,    275,   1760,    314,  94528,   7301,   9719,    456,
           306,    260,   4965,    260,  77262,    265,    512,    298, 194506,
           262,    260,    263,   5836,   4827,    346,    306,   5975,    348,
          1712,    288,    260,  48696,    305,    288,   6510,    305,    432,
        142642,    262,    515,   2477,   8278,  11475,    260,  15772,    345,
           631, 171427,  83975,  36668,    262, 104564,    261,  26976,    339,
           262,    260,  93666,  11170,    260,  75569,    289,    528,    314,
         77262,    265,    512,    298, 194506,    755,    306,   2106,    273,
           442,    261,   

In [15]:
from pprint import pprint

print("num_labels:", len(label2id))
pprint(label2id)           # label string -> id
# If you also want the reverse:
id2label = {i:l for l,i in label2id.items()}


num_labels: 59
{'B-account_pin': 1,
 'B-api_key': 2,
 'B-bank_routing_number': 3,
 'B-bban': 4,
 'B-company': 5,
 'B-credit_card_number': 6,
 'B-credit_card_security_code': 7,
 'B-customer_id': 8,
 'B-date': 9,
 'B-date_of_birth': 10,
 'B-date_time': 11,
 'B-driver_license_number': 12,
 'B-email': 13,
 'B-employee_id': 14,
 'B-first_name': 15,
 'B-iban': 16,
 'B-ipv4': 17,
 'B-ipv6': 18,
 'B-last_name': 19,
 'B-local_latlng': 20,
 'B-name': 21,
 'B-passport_number': 22,
 'B-password': 23,
 'B-phone_number': 24,
 'B-ssn': 25,
 'B-street_address': 26,
 'B-swift_bic_code': 27,
 'B-time': 28,
 'B-user_name': 29,
 'I-account_pin': 30,
 'I-api_key': 31,
 'I-bank_routing_number': 32,
 'I-bban': 33,
 'I-company': 34,
 'I-credit_card_number': 35,
 'I-credit_card_security_code': 36,
 'I-customer_id': 37,
 'I-date': 38,
 'I-date_of_birth': 39,
 'I-date_time': 40,
 'I-driver_license_number': 41,
 'I-email': 42,
 'I-employee_id': 43,
 'I-first_name': 44,
 'I-iban': 45,
 'I-ipv4': 46,
 'I-ipv6': 47,

In [16]:
# take one example from the processed train set
ex = train_processed[0]

tokens = tok.convert_ids_to_tokens(ex["input_ids"])
tag_ids = ex["labels"]
tag_str = [id2label[int(i)] for i in tag_ids if i != -100]
aligned_data = [(tokens[i], tag_ids[i], id2label[int(tag_ids[i])]) for i in range(len(tokens)) if tag_ids[i] != -100]

for t, tid, ts in aligned_data:
    print(f"{t:20s}  {tid:3d}  {ts}")

▁                       0  O
SUPP                    0  O
LY                      0  O
▁CHA                    0  O
IN                      0  O
▁MANA                   0  O
G                       0  O
EMENT                   0  O
▁                       0  O
AGRE                    0  O
EMENT                   0  O
▁This                   0  O
▁                       0  O
Supply                  0  O
▁Chain                  0  O
▁Management             0  O
▁                       0  O
Agreement               0  O
▁(                      0  O
the                     0  O
▁"                      0  O
Agreement               0  O
")                      0  O
▁is                     0  O
▁                       0  O
entered                 0  O
▁into                   0  O
▁as                     0  O
▁of                     0  O
▁this                   0  O
▁1                     38  I-date
st                     38  I-date
▁day                   38  I-date
▁of                    38  I

#Selecting a batch size and creating an iterator

In [17]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler
from keras.utils import pad_sequences
from sklearn.model_selection import train_test_split
from transformers import BertTokenizer, BertConfig
from torch.optim import AdamW
from transformers.optimization import get_linear_schedule_with_warmup
from transformers import BertForSequenceClassification
from tqdm import tqdm, trange  #for progress bars
import pandas as pd
import io
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image #for image rendering

In [18]:
train_processed

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 50346
})

In [19]:
# Select a batch size for training. For fine-tuning BERT on a specific task, the authors recommend a batch size of 16 or 32
batch_size = 32

# Create an iterator of our data with torch DataLoader. This helps save on memory during training because, unlike a for loop,
# with an iterator the entire dataset does not need to be loaded into memory

# train_data = TensorDataset(train_processed['input_ids'], train_processed['attention_mask'], train_processed['labels'])
train_sampler = RandomSampler(train_processed)
train_dataloader = DataLoader(train_processed, sampler=train_sampler, batch_size=batch_size, collate_fn=data_collator)

# validation_data = TensorDataset(val_processed)
validation_sampler = SequentialSampler(val_processed)
validation_dataloader = DataLoader(val_processed, sampler=validation_sampler, batch_size=batch_size, collate_fn=data_collator)


In [20]:
# Initializing a BERT bert-base-uncased style configuration
from transformers import AutoConfig, AutoModel, AutoTokenizer
model_name = "microsoft/deberta-v3-base"

configuration = AutoConfig.from_pretrained(model_name)

# Initializing a model from the bert-base-uncased style configuration
model = AutoModel.from_pretrained(model_name, config=configuration)

# Accessing the model configuration
# configuration = model.config
print(configuration)

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/371M [00:00<?, ?B/s]

DebertaV2Config {
  "attention_probs_dropout_prob": 0.1,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-07,
  "legacy": true,
  "max_position_embeddings": 512,
  "max_relative_positions": -1,
  "model_type": "deberta-v2",
  "norm_rel_ebd": "layer_norm",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "pooler_dropout": 0,
  "pooler_hidden_act": "gelu",
  "pooler_hidden_size": 768,
  "pos_att_type": [
    "p2c",
    "c2p"
  ],
  "position_biased_input": false,
  "position_buckets": 256,
  "relative_attention": true,
  "share_att_key": true,
  "transformers_version": "4.57.1",
  "type_vocab_size": 0,
  "vocab_size": 128100
}



In [21]:
# @title
from collections import Counter

used_ids = Counter()
for example in train_processed:
    used_ids.update(i for i in example["labels"] if i != -100)   # ignore masked

print("Unique label IDs used:", len(used_ids))
for lid, cnt in used_ids.most_common()[:30]:
    print(f"{lid:3d}  {id2label[lid]:30s}  {cnt}")

model.safetensors:   0%|          | 0.00/371M [00:00<?, ?B/s]

Unique label IDs used: 59
  0  O                               10605393
 50  I-name                          279838
 55  I-street_address                270382
 34  I-company                       177292
 38  I-date                          170379
 42  I-email                         73451
 53  I-phone_number                  32561
 57  I-time                          28504
 21  B-name                          23773
 47  I-ipv6                          20206
  5  B-company                       18286
 31  I-api_key                       17176
 45  I-iban                          11577
 26  B-street_address                9298
 13  B-email                         7530
  9  B-date                          7510
 33  I-bban                          7095
 37  I-customer_id                   7025
 56  I-swift_bic_code                6916
 39  I-date_of_birth                 6235
 52  I-password                      5671
 49  I-local_latlng                  5359
 43  I-employee_id            

In [22]:
model = AutoModelForTokenClassification.from_pretrained(
    "microsoft/mdeberta-v3-base",
    num_labels=len(bio_labels),
    id2label=id2label,
    label2id=label2id
)
model = nn.DataParallel(model)
model.to(device)

pytorch_model.bin:   0%|          | 0.00/1.33G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.33G [00:00<?, ?B/s]

Some weights of DebertaV2ForTokenClassification were not initialized from the model checkpoint at microsoft/mdeberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


DataParallel(
  (module): DebertaV2ForTokenClassification(
    (deberta): DebertaV2Model(
      (embeddings): DebertaV2Embeddings(
        (word_embeddings): Embedding(251000, 768, padding_idx=0)
        (LayerNorm): LayerNorm((768,), eps=1e-07, elementwise_affine=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (encoder): DebertaV2Encoder(
        (layer): ModuleList(
          (0-11): 12 x DebertaV2Layer(
            (attention): DebertaV2Attention(
              (self): DisentangledSelfAttention(
                (query_proj): Linear(in_features=768, out_features=768, bias=True)
                (key_proj): Linear(in_features=768, out_features=768, bias=True)
                (value_proj): Linear(in_features=768, out_features=768, bias=True)
                (pos_dropout): Dropout(p=0.1, inplace=False)
                (dropout): Dropout(p=0.1, inplace=False)
              )
              (output): DebertaV2SelfOutput(
                (dense): Linear(in_features=768, 

In [23]:
# Parametri ottimizzati per DaBERTa v3 su task di NER/PII detection
param_optimizer = list(model.named_parameters())

# Non applicare weight decay a bias e LayerNorm
no_decay = ['bias', 'LayerNorm.weight', 'LayerNorm.bias']

optimizer_grouped_parameters = [
    {
        'params': [p for n, p in param_optimizer if not any(nd in n for nd in no_decay)],
        'weight_decay': 0.01  # Valore tipico per DaBERTa (0.01-0.1)
    },
    {
        'params': [p for n, p in param_optimizer if any(nd in n for nd in no_decay)],
        'weight_decay': 0.0
    }
]

In [24]:
# Displaying a sample of the parameter_optimizer:  layer 3
layer_parameters = [p for n, p in model.named_parameters() if 'layer.3' in n]

In [25]:
# Displaying names of parameters for which weight decay is not applied
no_decay

['bias', 'LayerNorm.weight', 'LayerNorm.bias']

In [26]:
# Displaying the list of the two dictionaries
small_sample = [
    {'params': [p for n, p in param_optimizer if not any(nd in n for nd in no_decay)][:2],
     'weight_decay_rate': 0.01},
    {'params': [p for n, p in param_optimizer if any(nd in n for nd in no_decay)][:2],
     'weight_decay_rate': 0.0}
]

for i, group in enumerate(small_sample):
    print(f"Group {i+1}:")
    print(f"Weight decay rate: {group['weight_decay_rate']}")
    for j, param in enumerate(group['params']):
        print(f"Parameter {j+1}: {param}")

Group 1:
Weight decay rate: 0.01
Parameter 1: Parameter containing:
tensor([[ 0.0187, -0.0047, -0.0024,  ..., -0.0198,  0.0154,  0.0246],
        [-0.0253, -0.0084, -0.0109,  ..., -0.0201, -0.0056, -0.0047],
        [-0.0233, -0.0060, -0.0109,  ..., -0.0165, -0.0049, -0.0019],
        ...,
        [ 0.0157, -0.0065, -0.0094,  ..., -0.0145,  0.0058,  0.0148],
        [ 0.0123, -0.0027, -0.0048,  ..., -0.0144,  0.0147,  0.0226],
        [ 0.0246,  0.0043, -0.0025,  ..., -0.0212,  0.0232,  0.0247]],
       device='cuda:0', requires_grad=True)
Parameter 2: Parameter containing:
tensor([[ 0.0612, -0.1571,  0.1099,  ...,  0.0766, -0.1010, -0.0101],
        [-0.0350,  0.2230,  0.1971,  ...,  0.0668,  0.1351, -0.0952],
        [ 0.0621,  0.1041,  0.1530,  ..., -0.2688, -0.0279, -0.0012],
        ...,
        [ 0.2539,  0.0209,  0.0380,  ...,  0.1475,  0.0853,  0.0164],
        [-0.1096,  0.1230,  0.1782,  ...,  0.0779, -0.0202,  0.0695],
        [-0.1597,  0.1494,  0.2402,  ...,  0.1786, -0.02

#The hyperparameters for the training loop

In [27]:
# Number of training epochs (authors recommend between 2 and 4)
epochs = 4

optimizer = AdamW(optimizer_grouped_parameters,
                  lr = 2e-5, # args.learning_rate - default range: 1e-5 a 5e-5
                  eps = 1e-8 # args.adam_epsilon  - default is 1e-8.
                  )
# Total number of training steps is number of batches * number of epochs.
# `train_dataloader` contains batched data so `len(train_dataloader)` gives
# us the number of batches.
total_steps = len(train_dataloader) * epochs

# Create the learning rate scheduler.
scheduler = get_linear_schedule_with_warmup(optimizer,
                                            num_warmup_steps = 0, # Default value in run_glue.py
                                            num_training_steps = total_steps)

In [28]:
# Creating the Accuracy Measurement Function
# Function to calculate the accuracy of our predictions vs labels
def flat_accuracy(preds, labels):
    pred_flat = np.argmax(preds, axis=1).flatten()
    labels_flat = labels.flatten()
    return np.sum(pred_flat == labels_flat) / len(labels_flat)

def compute_metrics(preds, labels, label_list):
    """
    Metrics for PII detection (precision, recall, F1)
    """
    pred_flat = np.argmax(preds, axis=2).flatten()
    labels_flat = labels.flatten()

    # Rimuovi i padding tokens (assumendo che -100 sia usato per padding)
    mask = labels_flat != -100
    pred_flat = pred_flat[mask]
    labels_flat = labels_flat[mask]

    # F1 score (più importante dell'accuracy per PII detection!)
    f1 = f1_score(labels_flat, pred_flat, average='weighted')
    accuracy = np.sum(pred_flat == labels_flat) / len(labels_flat)

    return {
        'accuracy': accuracy,
        'f1': f1
    }

# training loop

In [29]:
import gc
import torch

# Prima del training
gc.collect()
torch.cuda.empty_cache()

# Verifica memoria disponibile
print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print(f"GPU memory reserved: {torch.cuda.memory_reserved() / 1024**3:.2f} GB")

GPU memory allocated: 1.04 GB
GPU memory reserved: 1.07 GB


In [30]:
def estimate_gpu_memory(model, batch_size, seq_length, num_labels):
    """
    Stima memoria GPU necessaria (approssimativa)
    """
    # Parametri del modello
    num_params = sum(p.numel() for p in model.parameters())

    # Memoria per i parametri (in GB)
    # FP32 = 4 bytes per parametro
    model_memory = (num_params * 4) / (1024**3)

    # Memoria per i gradienti (stessa dimensione dei parametri)
    gradient_memory = model_memory

    # Memoria per optimizer state (Adam = 2x parametri)
    optimizer_memory = model_memory * 2

    # Memoria per attivazioni (molto approssimativa)
    # DaBERTa: ~12 * num_layers * batch_size * seq_length * hidden_size * 4 bytes
    hidden_size = model.module.config.hidden_size # Access config through .module
    num_layers = model.module.config.num_hidden_layers # Access config through .module
    activation_memory = (12 * num_layers * batch_size * seq_length * hidden_size * 4) / (1024**3)

    total = model_memory + gradient_memory + optimizer_memory + activation_memory

    print("="*60)
    print("STIMA MEMORIA GPU (approssimativa)")
    print("="*60)
    print(f"Modello:                {model_memory:.2f} GB")
    print(f"Gradienti:              {gradient_memory:.2f} GB")
    print(f"Optimizer (Adam):       {optimizer_memory:.2f} GB")
    print(f"Attivazioni (forward):  {activation_memory:.2f} GB")
    print(f"-" * 60)
    print(f"TOTALE STIMATO:         {total:.2f} GB")
    print(f"\nBatch size:             {batch_size}")
    print(f"Sequence length:        {seq_length}")
    print(f"Parametri totali:       {num_params:,}")
    print("="*60)

    return total

# Usa così:
estimate_gpu_memory(model, batch_size=16, seq_length=512, num_labels=len(label2id))

STIMA MEMORIA GPU (approssimativa)
Modello:                1.04 GB
Gradienti:              1.04 GB
Optimizer (Adam):       2.07 GB
Attivazioni (forward):  3.38 GB
------------------------------------------------------------
TOTALE STIMATO:         7.52 GB

Batch size:             16
Sequence length:        512
Parametri totali:       278,264,123


7.521458551287651

In [31]:
import torch
import gc

def find_optimal_batch_size(model, device, max_seq_length=512, num_labels=None):
    """
    Trova il batch size ottimale tramite binary search
    """
    if num_labels is None:
        # Access config through .module for DataParallel
        num_labels = model.module.config.num_labels

    model.train()  # Importante: train mode usa più memoria

    def create_test_batch(batch_size):
        return {
            'input_ids': torch.randint(0, model.module.config.vocab_size, (batch_size, max_seq_length)).to(device),
            'attention_mask': torch.ones(batch_size, max_seq_length).to(device),
            'labels': torch.randint(-100, num_labels, (batch_size, max_seq_length)).to(device)
        }

    # Binary search - Start with a potentially smaller max_bs
    min_bs, max_bs = 1, 16  # Reduced initial max_bs
    optimal_bs = 1

    print("🔍 Cercando batch size ottimale...")
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memoria totale: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
    print(f"Sequence length: {max_seq_length}\n")

    while min_bs <= max_bs:
        batch_size = (min_bs + max_bs) // 2
        if batch_size == 0: # Avoid batch size 0
            break

        # Pulisci memoria
        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

        try:
            print(f"Testing batch_size={batch_size:2d}...", end=" ")

            batch = create_test_batch(batch_size)

            # Forward pass
            outputs = model(**batch)
            loss = outputs.loss

            # Backward pass (questo usa MOLTA più memoria!)
            loss.backward()

            # Misura memoria PEAK (massima usata)
            mem_allocated = torch.cuda.max_memory_allocated() / 1024**3
            mem_reserved = torch.cuda.max_memory_reserved() / 1024**3

            print(f"✅ OK | Peak: {mem_allocated:.2f}GB alloc, {mem_reserved:.2f}GB reserved")

            optimal_bs = batch_size
            min_bs = batch_size + 1 # Try larger batch size

        except RuntimeError as e:
            if "out of memory" in str(e).lower(): # Check for "out of memory" case-insensitively
                print(f"❌ OOM")
                max_bs = batch_size - 1 # Reduce batch size
            else:
                # Re-raise other RuntimeErrors
                raise e
        except Exception as e:
             # Catch any other exceptions during testing
             print(f"❌ Error: {e}")
             # Decide how to handle other errors, perhaps stop or reduce batch size
             max_bs = batch_size - 1 # Assuming other errors might also be memory related or blocking progress
        finally:
            # Cleanup
            if 'batch' in locals():
                del batch
            if 'outputs' in locals():
                del outputs
            if 'loss' in locals():
                del loss

            # Pulisci gradienti
            if model is not None:
                try:
                    model.zero_grad()
                except Exception as zero_grad_e:
                     print(f"Error during zero_grad: {zero_grad_e}")


            gc.collect()
            torch.cuda.empty_cache()

    print(f"\n{'='*70}")
    print(f"✨ RISULTATI:")
    print(f"{'='*70}")
    print(f"Batch size massimo testato con successo:              {optimal_bs}")
    print(f"Consigliato (safe, ~70%):        {max(1, int(optimal_bs * 0.7))}")
    print(f"Con gradient accumulation 2x:    batch={max(1, optimal_bs // 2)}, accum_steps=2")
    print(f"Con gradient accumulation 4x:    batch={max(1, optimal_bs // 4)}, accum_steps=4")
    print(f"Con gradient accumulation 8x:    batch={max(1, optimal_bs // 8)}, accum_steps=8")
    print(f"{'='*70}\n")

    return optimal_bs

# Esegui il test
gc.collect()
torch.cuda.empty_cache()

optimal_bs = find_optimal_batch_size(
    model,
    device,
    max_seq_length=512,
    num_labels=len(label2id)
)

🔍 Cercando batch size ottimale...
GPU: Tesla T4
Memoria totale: 14.74 GB
Sequence length: 512

Testing batch_size= 8... 

AcceleratorError: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
from sklearn.metrics import classification_report, f1_score
import numpy as np
import torch
from tqdm import trange

# Funzione metriche per NER
def compute_metrics(preds, labels):
    """
    Calcola metriche per PII detection (ignora padding -100)
    """
    pred_flat = np.argmax(preds, axis=2).flatten()
    labels_flat = labels.flatten()

    # Rimuovi padding
    mask = labels_flat != -100
    pred_flat = pred_flat[mask]
    labels_flat = labels_flat[mask]

    f1 = f1_score(labels_flat, pred_flat, average='weighted', zero_division=0)
    accuracy = np.sum(pred_flat == labels_flat) / len(labels_flat)

    return {'accuracy': accuracy, 'f1': f1}

# Store metriche
train_loss_set = []
val_loss_set = []
val_accuracy_set = []
val_f1_set = []

# Training loop
for epoch in trange(epochs, desc="Epoch"):

    # ============== TRAINING ==============
    model.train()

    tr_loss = 0
    nb_tr_steps = 0

    for step, batch in enumerate(train_dataloader):
        # ✅ Accedi ai dati come dizionario e sposta su GPU
        b_input_ids = batch['input_ids'].to(device)
        b_input_mask = batch['attention_mask'].to(device)
        b_labels = batch['labels'].to(device)

        # Zero gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(
            b_input_ids,
            attention_mask=b_input_mask,
            labels=b_labels
        )

        loss = outputs.loss

        # Backward pass
        loss.backward()

        # Gradient clipping (importante!)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        # Update
        optimizer.step()
        scheduler.step()

        # Tracking
        tr_loss += loss.item()
        nb_tr_steps += 1

        if step % 100 == 0 and step > 0:
            print(f"  Step {step}, Loss: {loss.item():.4f}")

    avg_train_loss = tr_loss / nb_tr_steps
    train_loss_set.append(avg_train_loss)
    print(f"\nEpoch {epoch + 1}")
    print(f"  Average training loss: {avg_train_loss:.4f}")


    # ============== VALIDATION ==============
    model.eval()

    eval_loss = 0
    nb_eval_steps = 0
    predictions, true_labels = [], []

    for batch in validation_dataloader:
        # ✅ Stesso pattern per validation
        b_input_ids = batch['input_ids'].to(device)
        b_input_mask = batch['attention_mask'].to(device)
        b_labels = batch['labels'].to(device)

        with torch.no_grad():
            outputs = model(
                b_input_ids,
                attention_mask=b_input_mask,
                labels=b_labels
            )

        loss = outputs.loss
        logits = outputs.logits

        eval_loss += loss.item()

        # Sposta su CPU
        logits = logits.detach().cpu().numpy()
        label_ids = b_labels.cpu().numpy()

        predictions.append(logits)
        true_labels.append(label_ids)
        nb_eval_steps += 1

    # Concatena batch
    predictions = np.concatenate(predictions, axis=0)
    true_labels = np.concatenate(true_labels, axis=0)

    # Calcola metriche
    metrics = compute_metrics(predictions, true_labels)

    avg_val_loss = eval_loss / nb_eval_steps
    val_loss_set.append(avg_val_loss)
    val_accuracy_set.append(metrics['accuracy'])
    val_f1_set.append(metrics['f1'])

    print(f"  Validation loss: {avg_val_loss:.4f}")
    print(f"  Validation Accuracy: {metrics['accuracy']:.4f}")
    print(f"  Validation F1: {metrics['f1']:.4f}")
    print()

print("\n" + "="*50)
print("TRAINING COMPLETED")
print("="*50)

In [ ]:
t = []

# Store our loss and accuracy for plotting
train_loss_set = []

# trange is a tqdm wrapper around the normal python range
for _ in trange(epochs, desc="Epoch"):

  # Training

  # Set our model to training mode (as opposed to evaluation mode)
  model.train()

  # Tracking variables
  tr_loss = 0
  nb_tr_examples, nb_tr_steps = 0, 0

  # Train the data for one epoch
  for step, batch in enumerate(train_dataloader):
    # Add batch to GPU
    batch = tuple(t.to(device) for t in batch)
    # Unpack the inputs from our dataloader
    b_input_ids, b_input_mask, b_labels = batch
    # Clear out the gradients (by default they accumulate)
    optimizer.zero_grad()
    # Forward pass
    outputs = model(b_input_ids, token_type_ids=None, attention_mask=b_input_mask, labels=b_labels)
    loss = outputs['loss']
    train_loss_set.append(loss.item())
    # Backward pass
    loss.backward()
    # Update parameters and take a step using the computed gradient
    optimizer.step()

    # Update the learning rate.
    scheduler.step()


    # Update tracking variables
    tr_loss += loss.item()
    nb_tr_examples += b_input_ids.size(0)
    nb_tr_steps += 1

  print("Train loss: {}".format(tr_loss/nb_tr_steps))

  # Validation

  # Put model in evaluation mode to evaluate loss on the validation set
  model.eval()

  # Tracking variables
  eval_loss, eval_accuracy = 0, 0
  nb_eval_steps, nb_eval_examples = 0, 0

  # Evaluate data for one epoch
  for batch in validation_dataloader:
    # Add batch to GPU
    batch = tuple(t.to(device) for t in batch)
    # Unpack the inputs from our dataloader
    b_input_ids, b_input_mask, b_labels = batch
    # Telling the model not to compute or store gradients, saving memory and speeding up validation
    with torch.no_grad():
      # Forward pass, calculate logit predictions
      logits = model(b_input_ids, token_type_ids=None, attention_mask=b_input_mask)

    # Move logits and labels to CPU
    logits = logits['logits'].detach().cpu().numpy()
    label_ids = b_labels.to('cpu').numpy()

    tmp_eval_accuracy = flat_accuracy(logits, label_ids)

    eval_accuracy += tmp_eval_accuracy
    nb_eval_steps += 1

  print("Validation Accuracy: {}".format(eval_accuracy/nb_eval_steps))